In [1]:
import sys
import os
import slurm_utils

HOME = os.environ['HOME']
BASECODE_PATH = os.path.join(HOME,'international-brain-lab/prior-localization/decoding')
if not BASECODE_PATH in sys.path:                                                                              
     sys.path.insert(0, BASECODE_PATH)

fb = open('slurm_decode_base.py','r')
filestr = fb.read()
fb.close()
exec(filestr)

fiteidstr = """
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
eid = sys.argv[1]
ADD_TO_SAVING_PATH = sys.argv[2]
print(eid)
if NULL_TYPE == 'impostor-session':
    IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA), 
                             '_'.join(['impostors',''])+'.pkl')
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)
else:
    impostordict = None
filenames = fit_eid(eid,sessdf,impostordict = impostordict)
print('saving to files:',filenames)
"""

ADD_TO_SAVING_PATH = 'testnulls2'
SUBDIRECTORY = dut.decoding_details(TARGET,MODEL,SCORE,
                             ESTIMATORSTR,
                             ALIGN_TIME,
                             CONTROL_FEATURES,
                             N_PSEUDO,NULL_TYPE,TIME_WINDOW,
                             ADD_TO_SAVING_PATH,
                             USE_FAKE_DATA=USE_FAKE_DATA)
SLURM_DIRECTORY = os.path.join(GROUP_HOME,'bensonb','international-brain-lab/prior-localization/slurm/')
OUTPUT_PATH = os.path.join(GROUP_HOME,'bensonb/international-brain-lab/prior-localization/decoding/')
print(SUBDIRECTORY)

IMPOSTOR_PATH = os.path.join(OUTPUT_PATH, 
                             SUBDIRECTORY, 
                             '_'.join(['impostors',''])+'.pkl')
if os.path.exists(IMPOSTOR_PATH):
    print('Impostor sessions already exist.')

decode_feedback_task_Logistic_accuracy_control_100_impostor-session_align_feedback_times_timeWin_0_0_2_testnulls2
Impostor sessions already exist.


In [3]:
# make python file
py_file = os.path.join(OUTPUT_PATH, SUBDIRECTORY, '_'.join(['slurm_decode_eid',''])+'.py')
if not os.path.exists(os.path.join(OUTPUT_PATH, SUBDIRECTORY)):
    os.mkdir(os.path.join(OUTPUT_PATH, SUBDIRECTORY))
with open(py_file,'w') as fp:
    fp.write(filestr)
    fp.write(fiteidstr)
    
sessdf = dut.query_sessions(selection=SESS_CRITERION)
sessdf = sessdf.sort_values('subject').set_index(['subject', 'eid'])
sessdf_eids = sessdf.index.unique(level='eid')

# sessdf_eids=np.unique(sessdf_eids)[30:32]
# sessdf_eids= ['dfd8e7df-dc51-4589-b6ca-7baccfeb94b4', '034e726f-b35f-41e0-8d6c-a22cc32391fb',
#             '7939711b-8b4d-4251-b698-b97c1eaa846e', 'fa704052-147e-46f6-b190-a65b837e605e',
#             '46794e05-3f6a-4d35-afb3-9165091a5a74', '1538493d-226a-46f7-b428-59ce5f43f0f9',
#             'b52182e7-39f6-4914-9717-136db589706e', '89f0d6ff-69f4-45bc-b89e-72868abb042a',
#             '3ce452b3-57b4-40c9-885d-1b814036e936', '2d5f6d81-38c4-4bdc-ac3c-302ea4d5f46e',
#             '4b7fbad4-f6de-43b4-9b15-c7c7ef44db4b', 'd839491f-55d8-4cbe-a298-7839208ba12b',
#             'd2918f52-8280-43c0-924b-029b2317e62c', 'c99d53e6-c317-4c53-99ba-070b26673ac4',
#             'ecb5520d-1358-434c-95ec-93687ecd1396', '5386aba9-9b97-4557-abcd-abc2da66b863',
#             '4b00df29-3769-43be-bb40-128b1cba6d35', '3663d82b-f197-4e8b-b299-7b803a155b84',
#             '85dc2ebd-8aaf-46b0-9284-a197aee8b16f', '15f742e1-1043-45c9-9504-f1e8a53c1744']

# collect all target vectors for impostor sessions
if not os.path.exists(IMPOSTOR_PATH):
    fimpostor = open(IMPOSTOR_PATH, 'wb')
    impostordict = {}
    n_eid = 0
    for eid in sessdf_eids:
        print(n_eid, len(sessdf_eids), eid)
        n_eid += 1
        impostordict[eid] = get_target_vector_eid(eid, sessdf)
    pickle.dump(impostordict, fimpostor)
    fimpostor.close()
else:
    with open(IMPOSTOR_PATH,'rb') as fimpostor:
        impostordict = pickle.load(fimpostor)

fit_metadata['submitted_eids'] = sessdf_eids
fit_metadata['impostor_dictionary'] = impostordict
fn = os.path.join(OUTPUT_PATH,SUBDIRECTORY,'DATE_results')
fn = fn + '.parquet'
metadata_df = pd.Series({'filename': fn, **fit_metadata})
metadata_fn = '.'.join([fn.split('.')[0], 'metadata', 'pkl'])
metadata_fn = metadata_fn.replace('DATE',str(date.today()))
metadata_df.to_pickle(metadata_fn)
print('created metadata file and impostor sessions')

Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/tmp7yaqr3w2/cache.zip Bytes: 99070982


1256698886it [00:10, 119160616.51it/s]                                                                                 


created metadata file and impostor sessions


In [6]:
eid = '15f742e1-1043-45c9-9504-f1e8a53c1744'
one = ONE()  # mode='local'
atlas = AllenAtlas()

estimator = ESTIMATOR #(**ESTIMATOR_KWARGS)

subject = sessdf.xs(eid, level='eid').index[0]
subjeids = sessdf.xs(subject, level='subject').index.unique()
brainreg = dut.BrainRegions()
behavior_data = mut.load_session(eid, one=one)
pLeft_vec = np.array(behavior_data['probabilityLeft'])
try:
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, beh_data=behavior_data,
                              one=one)
except ValueError:
    print('Model not fit.')
    tvec = dut.compute_target(TARGET, subject, subjeids, eid, MODELFIT_PATH,
                              modeltype=MODEL, one=one)

fvecs = [dut.compute_target(feature, subject, subjeids, eid, MODELFIT_PATH,
                          modeltype=MODEL, beh_data=behavior_data,
                          one=one) for feature in CONTROL_FEATURES]

try:
    trialsdf = bbone.load_trials_df(eid, one=one, addtl_types=['firstMovement_times'])
    if len(trialsdf) != len(tvec):
        raise IndexError
except IndexError:
    raise IndexError('Problem in the dimensions of dataframe of session')
trialsdf['react_times'] = trialsdf['firstMovement_times'] - trialsdf['goCue_times']

local md5 or size mismatch on dataset: cortexlab/Subjects/KS022/2019-12-09/001alf/_ibl_trials.response_times.npy


Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/cortexlab/Subjects/KS022/2019-12-09/001/alf/_ibl_trials.response_times.e9ab5a29-9db4-4e8b-ac1f-ec82fdff65c0.npy Bytes: 6736


100%|█████████████████████████████████████████████████████████████████████████| 6736/6736 [00:00<00:00, 1137896.48it/s]
local md5 or size mismatch on dataset: cortexlab/Subjects/KS022/2019-12-09/001alf/_ibl_trials.response_times.npy


Downloading: /home/groups/sganguli/bensonb/Downloads/ONE/cache_dir/cortexlab/Subjects/KS022/2019-12-09/001/alf/_ibl_trials.response_times.e9ab5a29-9db4-4e8b-ac1f-ec82fdff65c0.npy Bytes: 6736


100%|█████████████████████████████████████████████████████████████████████████| 6736/6736 [00:00<00:00, 2523024.80it/s]


In [7]:
trialsdf.keys()

Index(['choice', 'probabilityLeft', 'feedbackType', 'feedback_times',
       'contrastLeft', 'contrastRight', 'goCue_times', 'stimOn_times',
       'firstMovement_times', 'trial_start', 'trial_end', 'react_times'],
      dtype='object')

In [12]:
def generate_shifts():
    out = np.arange(-10,10)
    np.random.shuffle(out)
    i = 0
    while i < len(out):
        yield out[i]
        i+=1
        
gs = generate_shifts()
for j in range(30):
    print(next(generate_shifts()))
    

-6
-1
-8
1
-1
7
4
5
7
-10
-5
-3
-5
-9
-1
6
-3
9
0
5
9
8
-7
-1
-3
9
-4
-10
9
1


In [4]:
# submit python file with eid inputs
for eid in sessdf_eids:
    slurm_utils.submit_python_file(py_file, 
                             eid, 
                             ADD_TO_SAVING_PATH = ADD_TO_SAVING_PATH,
                             n_gigs_ram = 8,
                             SLURM_DIRECTORY = SLURM_DIRECTORY,
                             SUBDIRECTORY = SUBDIRECTORY)
    

Submitted batch job 48265616
Submitted batch job 48265617
Submitted batch job 48265618
Submitted batch job 48265619
Submitted batch job 48265620
Submitted batch job 48265622
Submitted batch job 48265623
Submitted batch job 48265624
Submitted batch job 48265625
Submitted batch job 48265626
Submitted batch job 48265627
Submitted batch job 48265628
Submitted batch job 48265629
Submitted batch job 48265630
Submitted batch job 48265631
Submitted batch job 48265632
Submitted batch job 48265633
Submitted batch job 48265634
Submitted batch job 48265635
Submitted batch job 48265636
Submitted batch job 48265637
Submitted batch job 48265638
Submitted batch job 48265639
Submitted batch job 48265640
Submitted batch job 48265641
Submitted batch job 48265642
Submitted batch job 48265643
Submitted batch job 48265644
Submitted batch job 48265645
Submitted batch job 48265646
Submitted batch job 48265647
Submitted batch job 48265648
Submitted batch job 48265649
Submitted batch job 48265650
Submitted batc

Submitted batch job 48265987
Submitted batch job 48265988
Submitted batch job 48265989
Submitted batch job 48265990
Submitted batch job 48265991
Submitted batch job 48265992
Submitted batch job 48265997
Submitted batch job 48265998
Submitted batch job 48265999
Submitted batch job 48266000
Submitted batch job 48266001
Submitted batch job 48266002
Submitted batch job 48266003
Submitted batch job 48266004
Submitted batch job 48266005
Submitted batch job 48266006
Submitted batch job 48266007
Submitted batch job 48266008
Submitted batch job 48266009
Submitted batch job 48266010
Submitted batch job 48266011
Submitted batch job 48266012
Submitted batch job 48266013
Submitted batch job 48266014
Submitted batch job 48266015
Submitted batch job 48266016
Submitted batch job 48266017
Submitted batch job 48266018
Submitted batch job 48266019
Submitted batch job 48266020
Submitted batch job 48266021
Submitted batch job 48266022
Submitted batch job 48266023
Submitted batch job 48266024
Submitted batc

In [2]:

os.system("squeue -u $USER");
slurm_utils.get_decoding_output_files(SLURM_DIRECTORY = SLURM_DIRECTORY,
                                      SUBDIRECTORY = SUBDIRECTORY)

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          48265690    normal /home/gr  bensonb  R    9:38:55      1 sh03-01n60
          48265691    normal /home/gr  bensonb  R    9:38:55      1 sh03-01n60
          48265989    normal /home/gr  bensonb  R    7:24:18      1 sh02-01n19
          48281679    normal interact  bensonb  R       9:58      1 sh02-01n48


['/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_feedback_task_Logistic_accuracy_control_100_impostor-session_align_feedback_times_timeWin_0_0_2_testnulls2/SWC_038/1e45d992-c356-40e1-9be1-a506d944896f/probe00/2022-03-30_APN.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_feedback_task_Logistic_accuracy_control_100_impostor-session_align_feedback_times_timeWin_0_0_2_testnulls2/SWC_038/1e45d992-c356-40e1-9be1-a506d944896f/probe00/2022-03-30_DG.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_feedback_task_Logistic_accuracy_control_100_impostor-session_align_feedback_times_timeWin_0_0_2_testnulls2/SWC_038/1e45d992-c356-40e1-9be1-a506d944896f/probe01/2022-03-30_ACB.pkl',
 '/home/groups/sganguli/bensonb/international-brain-lab/prior-localization/decoding/decode_feedback_task_Logistic_accuracy_control_100_impostor-session_align_feedback_times_timeWin_0_0_

In [3]:
#SUBDIRECTORY = 'decode_pLeft_task_Logistic_accuracy_control_10_impostor-session_align_goCue_times_timeWin_-0_4_-0_1_testnulltype1'

slurm_utils.gather_save_outputs(SUBDIRECTORY, SLURM_DIRECTORY, OUTPUT_PATH)